# SteelOpt — material recovery & nesting decision-support proof of concept

**Synthetic data only · not a STRUMIS integration · not CNC-ready · no demonstrated client savings**

**Question being tested:** Can transparent material checks and simple placement logic identify *potential* improvements worth validating on real historical nesting jobs?

This notebook shows a reproducible rectangular-plate example, matches offcuts to downstream demand, and records the gating rules that prevent unsafe or unauthorized allocation. The proposed production pilot would benchmark against actual manual plans and STRUMIS auto-nesting output on matched jobs; **neither is available here**.


## 1 · Load demonstrator inventory and jobs

In [ ]:
from pathlib import Path
import sys, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
root = Path.cwd()
if not (root/'src').exists():
    raise RuntimeError('Open notebook from repository root: jupyter notebook demo.ipynb')
sys.path.insert(0,str(root))
from src.steelopt import Stock, Part, pack_shelves, validate_layout, layout_stats, offcut_matches, plot_layout
stock_df=pd.read_csv(root/'data/stocks.csv')
parts_df=pd.read_csv(root/'data/parts.csv')
stocks=[Stock(**r) for r in stock_df.to_dict('records')]
parts=[Part(**r) for r in parts_df.to_dict('records')]
display(stock_df,parts_df)

## 2 · Compare two illustrative heuristics, **not** client manual vs STRUMIS

Both plans are generated by this notebook; the first processes demand in input order, while the second sorts it by longest edge. They are **synthetic algorithmic examples**, not historical manual or STRUMIS results. A plan with missing parts must not be interpreted as superior just because its utilization is higher. Material costs cannot be inferred from the percentage alone.


In [ ]:
plate=next(s for s in stocks if s.id=='PL-A01')
demand=[p for p in parts if p.project=='A']
input_plan,input_unplaced=pack_shelves(plate,demand,kerf=5,sort=False)
sorted_plan,sorted_unplaced=pack_shelves(plate,demand,kerf=5,sort=True)
results=pd.DataFrame([dict(method='Input-order shelf',**layout_stats(plate,input_plan),unplaced=', '.join(input_unplaced) or 'None'),dict(method='Sorted shelf',**layout_stats(plate,sorted_plan),unplaced=', '.join(sorted_unplaced) or 'None')])
display(results)
fig,axs=plt.subplots(1,2,figsize=(13,4))
plot_layout(axs[0],plate,input_plan,'Synthetic input-order heuristic')
plot_layout(axs[1],plate,sorted_plan,'Synthetic sorted heuristic')
plt.tight_layout(); plt.savefig(root/'outputs/nesting-comparison.png',dpi=130,bbox_inches='tight'); plt.show()
assert not validate_layout(plate,demand,input_plan)
assert not validate_layout(plate,demand,sorted_plan)

## 3 · Offcut matching (recommendations only)

Matches require same steel grade and thickness, stock marked **available** and **traceable**, geometric clearance with 5 mm illustrative kerf, and allowed orientation. **Project ownership is never changed.** Cross-project matches must be approved by Cost Control and the nesting/yard owners, including cost transfer and physical traceability. Values shown are *illustrative area-value proxies*, **not** avoided purchase orders, realized savings or client steel prices.


In [ ]:
needed=[p for p in parts if p.project=='B']
match_rows=offcut_matches(needed,stocks,kerf=5)
match_df=pd.DataFrame(match_rows)
display(match_df)
assert len(match_df)>0
assert match_df['decision'].eq('PENDING_COST_CONTROL_APPROVAL').all()
assert match_df['realized_savings'].isna().all()

## 4 · Deliberately rejected cases

These controls are *more important* than an attractive visualization: mismatched thickness, quarantined stock, missing traceability, and a part which does not fit after kerf allowance must be excluded. A physically valid cross-project match is still **not** automatically actionable.


In [ ]:
examples=[]
for id_,description in [('OC-A01','Eligible, cost approval still needed'),('OC-A02','Wrong thickness'),('OC-A03','Quarantined'),('OC-A04','Untraceable'),('OC-A05','Wrong grade')]:
    item=next(s for s in stocks if s.id==id_)
    target=next(p for p in needed if p.id=='P06')
    matches=offcut_matches([target],[item],kerf=5)
    examples.append({'stock_id':id_,'case':description,'geometric_material_match':bool(matches),'decision':matches[0]['decision'] if matches else 'REJECTED'})
display(pd.DataFrame(examples))
assert sum(row['geometric_material_match'] for row in examples)==1

## 5 · What would turn this into a credible client pilot?

1. Confirm actual STRUMIS/PLM export fields, licenses, and revision identifiers; never assume an API exists.
2. Collect real approved drawings, material grade/thickness, stock dimensions, offcut genealogy, reservations, CNC constraints, and approved/actual cut outcomes.
3. Build a **matched-job** baseline: same parts, same allowable stock, same deadlines and machine assumptions; compare current manual and native auto-nesting fairly.
4. Report **total input steel mass, delivered part mass, true scrap mass, reusable remnant mass, stock purchases and time** separately; account for open inventory and rework.
5. Require feasibility review by nesting/sizing engineering; cross-project transfers require Cost Control + inventory controls. Run prospectively in shadow mode; do not feed output straight into a CNC.
6. Gate extension on *verified* savings net of labor, rework, handling, delay and integration costs. Stop if the data cannot trace material or if any unacceptable safety/traceability violation appears.

### Model limitations
Rectangles only; ignores arbitrary structural geometry, bridges/micro-joints, machine table details, heat/lot restrictions, edge quality, grain/rolling directions, remnant shape damage, real kerf/tool paths, handling, delivery constraints, revision validity and per-project commercial exceptions. The conservative clearance check is still **not an engineering sign-off**. Avoid using this for real cut instructions.
